Read-only audit

In [1]:
from pathlib import Path

import pandas as pd
from IPython.display import display

data_dir = next(
    (folder / "Data" for folder in (Path.cwd(), Path.cwd().parent)
     if (folder / "Data").is_dir()),
    None,
)
if data_dir is None:
    raise FileNotFoundError("Open the notebook from the project root or its notebooks folder.")

season_files = {
    "2010/11": "E0 (29).csv",
    "2011/12": "E0 (28).csv",
    "2012/13": "E0 (27).csv",
    "2013/14": "E0 (26).csv",
    "2014/15": "E0 (25).csv",
    "2015/16": "E0 (19).csv",
    "2016/17": "E0 (20).csv",
    "2017/18": "E0 (21).csv",
    "2018/19": "E0 (22).csv",
    "2019/20": "E0 (23).csv",
    "2020/21": "E0 (24).csv",
}

In [2]:
def inspect_season(season):
    filename = season_files[season]
    raw = pd.read_csv(
        data_dir / filename, dtype="string", keep_default_na=False, skip_blank_lines=False
    )
    blank = raw.apply(lambda column: column.str.strip().eq(""))
    populated = ~blank.all(axis=1)
    blank_counts = blank.loc[populated].sum()

    date_shapes = raw.loc[populated, "Date"].str.replace(r"\d", "#", regex=True)
    date_formats = date_shapes.map({"##/##/##": "DD/MM/YY", "##/##/####": "DD/MM/YYYY"})
    formats = ", ".join(date_formats.fillna("other or blank").unique())

    summary = {
        "Season": season,
        "File": filename,
        "Raw records": len(raw),
        "Populated records": int(populated.sum()),
        "Empty records": int((~populated).sum()),
        "Columns": raw.shape[1],
        "Date format": formats,
        "Kickoff column": "present" if "Time" in raw.columns else "absent",
        "Blank cells": int(blank_counts.sum()),
    }
    return raw, summary, blank_counts


sample_seasons = ("2014/15", "2019/20")
audits = {season: inspect_season(season) for season in sample_seasons}
display(pd.DataFrame([audits[season][1] for season in sample_seasons]).set_index("Season"))

,File,Raw records,Populated records,Empty records,Columns,Date format,Kickoff column,Blank cells
Season,,,,,,,,
2014/15,E0 (25).csv,381,380,1,68,DD/MM/YY,absent,1023
2019/20,E0 (23).csv,380,380,0,106,DD/MM/YYYY,present,0


In [3]:
for season in sample_seasons:
    raw, summary, _ = audits[season]
    columns = ["Date", "HomeTeam", "AwayTeam", "FTHG", "FTAG", "FTR"]
    if "Time" in raw.columns:
        columns.insert(1, "Time")
    print(f"{season} | {summary['File']} | first two raw records")
    display(raw.loc[:, columns].head(2))

2014/15 | E0 (25).csv | first two raw records


,Date,HomeTeam,AwayTeam,FTHG,FTAG,FTR
0,16/08/14,Arsenal,Crystal Palace,2,1,H
1,16/08/14,Leicester,Everton,2,2,D


2019/20 | E0 (23).csv | first two raw records


,Date,Time,HomeTeam,AwayTeam,FTHG,FTAG,FTR
0,09/08/2019,20:00,Liverpool,Norwich,4,1,H
1,10/08/2019,12:30,West Ham,Man City,0,5,A


In [4]:
older_columns = set(audits["2014/15"][0].columns)
newer_columns = set(audits["2019/20"][0].columns)

schema_counts = pd.Series({
    "Shared columns": len(older_columns & newer_columns),
    "Only in 2014/15": len(older_columns - newer_columns),
    "Only in 2019/20": len(newer_columns - older_columns),
}, name="Column count")
display(schema_counts.to_frame())

sample_blanks = pd.concat(
    {season: audits[season][2] for season in sample_seasons},
    names=["Season", "Column"],
)
display(sample_blanks[sample_blanks.gt(0)].to_frame("Blank cells in populated records"))

,Column count
Shared columns,44
Only in 2014/15,24
Only in 2019/20,62


Blank cells in populated records
Season  Column                                  
2014/15 IWH                                    1
        IWD                                    1
        IWA                                    1
        SJH                                  340
        SJD                                  340
        SJA                                  340

The two files each contain 380 populated records. The 2014/15 file has one additional entirely blank record, and its blank cells within populated records are in betting fields(not important). The 2019/20 file supplies kickoff times and has zero blank cells in its supplied columns.

In [5]:
for season in season_files:
    if season not in audits:
        audits[season] = inspect_season(season)

inventory = pd.DataFrame([audits[season][1] for season in season_files]).set_index("Season")
display(inventory)

totals = inventory[["Raw records", "Populated records", "Empty records", "Blank cells"]].sum()
display(totals.to_frame("Across 11 seasons"))

,File,Raw records,Populated records,Empty records,Columns,Date format,Kickoff column,Blank cells
Season,,,,,,,,
2010/11,E0 (29).csv,380,380,0,71,DD/MM/YY,absent,0
2011/12,E0 (28).csv,380,380,0,71,DD/MM/YY,absent,0
2012/13,E0 (27).csv,380,380,0,74,DD/MM/YY,absent,6
2013/14,E0 (26).csv,380,380,0,68,DD/MM/YY,absent,60
2014/15,E0 (25).csv,381,380,1,68,DD/MM/YY,absent,1023
2015/16,E0 (19).csv,380,380,0,65,DD/MM/YYYY,absent,3
2016/17,E0 (20).csv,380,380,0,65,DD/MM/YY,absent,0
2017/18,E0 (21).csv,380,380,0,65,DD/MM/YYYY,absent,0
2018/19,E0 (22).csv,380,380,0,62,DD/MM/YYYY,absent,0


,Across 11 seasons
Raw records,4181
Populated records,4180
Empty records,1
Blank cells,1092


In [6]:
blank_counts_by_season = pd.DataFrame(
    {season: audits[season][2] for season in season_files}
).T
seasons_with_column = blank_counts_by_season.notna().sum()
print(f"{len(seasons_with_column)} columns appear at least once.")
print(f"{seasons_with_column.eq(len(season_files)).sum()} columns appear in all 11 files.")

focus_columns = ["Div", "Date", "HomeTeam", "AwayTeam", "FTHG", "FTAG", "FTR", "Time"]
coverage = pd.DataFrame({
    "Seasons with column": seasons_with_column,
    "Blank cells where supplied": blank_counts_by_season.sum(min_count=1).astype("Int64"),
})
display(coverage.reindex(focus_columns).rename_axis("Source column"))

139 columns appear at least once.
38 columns appear in all 11 files.


,Seasons with column,Blank cells where supplied
Source column,,
Div,11,0
Date,11,0
HomeTeam,11,0
AwayTeam,11,0
FTHG,11,0
FTAG,11,0
FTR,11,0
Time,2,0


The inventory shows 4,181 raw records and 4,180 populated records. we need to account for the one entirely blank record explicitly when we clean the files.
The shared fixture fields in the coverage table are complete. Date values appear in two day-first formats, so date conversion will need to handle both.
Kickoff times are supplied in two seasons. Their absence in the other nine is a difference in what was collected.